# GraphRAG Notebook Driver
This notebook uses a LangGraph workflow (`graphrag/graphrag_langgraph.py`) for hybrid Cypher + vector retrieval over Neo4j.

In [1]:
# Run once in this venv if needed:
# %pip install langgraph langchain langchain-openai neo4j numpy pandas python-dotenv


In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ").strip()

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

Enter OPENAI_API_KEY:  ········


OPENAI_API_KEY set: True


In [3]:
from graphrag_helpers import GraphRAGAssistant

NEO4J_ENV_FILE = "Neo4j-d2f72502-Created-2026-03-01.txt"

rag = GraphRAGAssistant.from_neo4j_env_file(
    NEO4J_ENV_FILE,
    model="gpt-4o-mini",
    embeddings_model="text-embedding-3-small",
)
print("GraphRAG assistant initialized.")

GraphRAG assistant initialized.


In [5]:
rag.build_vector_index(force_rebuild=False)
print("Hybrid vector index ready.")

Hybrid vector index ready.


In [6]:
def ask_graphrag(
    question: str,
    max_retries: int = 2,
    interactive_clarification: bool = True,
    clarification_text: str | None = None,
):
    callback = None
    if clarification_text:
        used = {"done": False}
        def _one_shot_callback(clarification_question: str, prior_result: dict):
            if not used["done"]:
                used["done"] = True
                return clarification_text
            return None
        callback = _one_shot_callback
        interactive_clarification = True
    if interactive_clarification:
        result = rag.ask_with_user_clarification(
            question,
            max_retries=max_retries,
            top_n_default=5,
            k_vector=6,
            max_clarification_rounds=2,
            clarification_callback=callback,
        )
    else:
        result = rag.ask(question, max_retries=max_retries, top_n_default=5, k_vector=6)
    print(result["answer"])
    return result


In [7]:
# Example:
# ask_graphrag("For Singapore in 2024, show top import/export partners and explain whether it behaves like a shadow hub.")

In [7]:
ask_graphrag("For Singapore in 2024, show top import/export partners and explain whether it behaves like a shadow hub.")

/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QuestionPlan(intent='gene...nd shadow hub behavior'), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(


Clarification needed: Do you want to know the top import/export partners for Singapore in 2024 and whether it behaves like a shadow hub?


/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:317: RuntimeWarning: divide by zero encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:317: RuntimeWarning: overflow encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:317: RuntimeWarning: invalid value encountered in matmul
  sims = (self.matrix @ q_vec) / denom


Would you like to add clarification and retry? [y/N]:  y
Enter clarification:  Yes


Clarification needed: Please specify if you want the import/export partners or the shadow hub behavior explanation.


Would you like to add clarification and retry? [y/N]:  Y
Enter clarification:  Yes


/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QualityReport(is_sufficie...ne, retrieval_gap=False), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(
/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=RewritePlan(refined_quest...w hub characteristics."), input_type=RewritePlan])
  return self.__pydantic_serializer__.to_python(
/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QuestionPlan(intent='hub_...

In 2024, Singapore's top export partners are Indonesia, Malaysia, Australia, China, and Vietnam, with trade values of approximately $10.99 billion, $10.76 billion, $6.84 billion, $3.73 billion, and $2.61 billion, respectively [C1]. On the import side, Singapore's leading partners are India, Malaysia, Brazil, the USA, and China, with trade values of about $12.88 billion, $12.87 billion, $10.68 billion, $9.20 billion, and $7.85 billion, respectively [C2].

These trade relationships illustrate Singapore's role as a shadow hub in global trade. The metrics for Singapore in 2024 indicate a shadow residual of 0.055, a shadow rank of 4, and a betweenness of 0.109 [C3]. A high shadow residual suggests that Singapore is unusually central in the trade network relative to its trade volume, while the betweenness metric indicates a significant role in connecting various trade routes. The combination of these metrics suggests that Singapore behaves like a shadow hub in the structural sense used in th

{'answer': "In 2024, Singapore's top export partners are Indonesia, Malaysia, Australia, China, and Vietnam, with trade values of approximately $10.99 billion, $10.76 billion, $6.84 billion, $3.73 billion, and $2.61 billion, respectively [C1]. On the import side, Singapore's leading partners are India, Malaysia, Brazil, the USA, and China, with trade values of about $12.88 billion, $12.87 billion, $10.68 billion, $9.20 billion, and $7.85 billion, respectively [C2].\n\nThese trade relationships illustrate Singapore's role as a shadow hub in global trade. The metrics for Singapore in 2024 indicate a shadow residual of 0.055, a shadow rank of 4, and a betweenness of 0.109 [C3]. A high shadow residual suggests that Singapore is unusually central in the trade network relative to its trade volume, while the betweenness metric indicates a significant role in connecting various trade routes. The combination of these metrics suggests that Singapore behaves like a shadow hub in the structural se

In [7]:
ask_graphrag("Across all years, which country is the top shadow hub?")

/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QuestionPlan(intent='top_...one, retrieval_focus=''), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: divide by zero encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: overflow encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: invalid value encountered in matmul
  sims = (self.matrix @ q_vec) / denom


The top shadow hub across all years is the United States (USA), which has a shadow residual of 0.3052 and a shadow rank of 1.0 in 2023, indicating a significantly high intermediary importance in the oil-trade network relative to its trade volume [C1].

Citations:
- [C1] Top shadow hub across all years | Neo4j SHADOW_HUB relationship | all years, rows=1
- [V1] Vector hit shadow::GBR::2020 | shadow_metric | score=0.655
- [V2] Vector hit shadow::GBR::2024 | shadow_metric | score=0.649
- [V3] Vector hit shadow::GBR::2019 | shadow_metric | score=0.648
- [V4] Vector hit shadow::USA::2022 | shadow_metric | score=0.647
- [V5] Vector hit shadow::USA::2020 | shadow_metric | score=0.646
- [V6] Vector hit shadow::GBR::2022 | shadow_metric | score=0.645


/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QualityReport(is_sufficie...ne, retrieval_gap=False), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(


{'answer': 'The top shadow hub across all years is the United States (USA), which has a shadow residual of 0.3052 and a shadow rank of 1.0 in 2023, indicating a significantly high intermediary importance in the oil-trade network relative to its trade volume [C1].\n\nCitations:\n- [C1] Top shadow hub across all years | Neo4j SHADOW_HUB relationship | all years, rows=1\n- [V1] Vector hit shadow::GBR::2020 | shadow_metric | score=0.655\n- [V2] Vector hit shadow::GBR::2024 | shadow_metric | score=0.649\n- [V3] Vector hit shadow::GBR::2019 | shadow_metric | score=0.648\n- [V4] Vector hit shadow::USA::2022 | shadow_metric | score=0.647\n- [V5] Vector hit shadow::USA::2020 | shadow_metric | score=0.646\n- [V6] Vector hit shadow::GBR::2022 | shadow_metric | score=0.645',
 'plan': {'intent': 'top_shadow_hubs_all_years',
  'country_name': None,
  'country_iso3': None,
  'year': None,
  'top_n': 5,
  'needs_clarification': False,
  'clarification_question': None,
  'retrieval_focus': ''},
 'quali

In [7]:
ask_graphrag("Which country has the highest betweenness in 2024?")

/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QuestionPlan(intent='high...one, retrieval_focus=''), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: divide by zero encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: overflow encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: invalid value encountered in matmul
  sims = (self.matrix @ q_vec) / denom


In 2024, the country with the highest betweenness in the oil-trade network is the United States (USA), with a betweenness score of approximately 0.3495. This score indicates a significant intermediary role within the network, suggesting that the USA behaves like a shadow hub in the structural sense used in this project, as it has a high shadow residual of 0.2243 and a shadow rank of 1.0, reflecting its centrality relative to trade volume [C1].

Citations:
- [C1] Country with highest betweenness | Neo4j SHADOW_HUB relationship | year=2024, rows=1
- [V1] Vector hit shadow::USA::2024 | shadow_metric | score=0.459
- [V2] Vector hit shadow::GBR::2024 | shadow_metric | score=0.459
- [V3] Vector hit shadow::HKG::2024 | shadow_metric | score=0.458
- [V4] Vector hit shadow::JPN::2024 | shadow_metric | score=0.457
- [V5] Vector hit shadow::JPN::2023 | shadow_metric | score=0.452
- [V6] Vector hit shadow::USA::2023 | shadow_metric | score=0.451


/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QualityReport(is_sufficie...ne, retrieval_gap=False), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(


{'answer': 'In 2024, the country with the highest betweenness in the oil-trade network is the United States (USA), with a betweenness score of approximately 0.3495. This score indicates a significant intermediary role within the network, suggesting that the USA behaves like a shadow hub in the structural sense used in this project, as it has a high shadow residual of 0.2243 and a shadow rank of 1.0, reflecting its centrality relative to trade volume [C1].\n\nCitations:\n- [C1] Country with highest betweenness | Neo4j SHADOW_HUB relationship | year=2024, rows=1\n- [V1] Vector hit shadow::USA::2024 | shadow_metric | score=0.459\n- [V2] Vector hit shadow::GBR::2024 | shadow_metric | score=0.459\n- [V3] Vector hit shadow::HKG::2024 | shadow_metric | score=0.458\n- [V4] Vector hit shadow::JPN::2024 | shadow_metric | score=0.457\n- [V5] Vector hit shadow::JPN::2023 | shadow_metric | score=0.452\n- [V6] Vector hit shadow::USA::2023 | shadow_metric | score=0.451',
 'plan': {'intent': 'highest_

In [8]:
ask_graphrag("Give me the country with the greatest shadow residual over all years")

/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QuestionPlan(intent='top_...eatest shadow residual'), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(
/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QualityReport(is_sufficie...one, retrieval_gap=True), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(
/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=RewritePlan(refined_quest..

The country with the highest shadow residual when comparing data across all available years is the United States (USA), with a shadow residual of 0.3052 in the year 2023. This indicates that the USA exhibits a significant level of intermediary importance in the oil-trade network relative to its trade volume, as reflected by its shadow rank of 1.0 and a betweenness score of 0.4859, suggesting a central role in trade relationships [C1].

Citations:
- [C1] Top shadow hub across all years | Neo4j SHADOW_HUB relationship | all years, rows=1
- [V1] Vector hit shadow::GBR::2024 | shadow_metric | score=0.621
- [V2] Vector hit shadow::GBR::2022 | shadow_metric | score=0.621
- [V3] Vector hit shadow::GBR::2019 | shadow_metric | score=0.621
- [V4] Vector hit shadow::GBR::2023 | shadow_metric | score=0.620
- [V5] Vector hit shadow::GBR::2020 | shadow_metric | score=0.618
- [V6] Vector hit shadow::GBR::2021 | shadow_metric | score=0.615


{'answer': 'The country with the highest shadow residual when comparing data across all available years is the United States (USA), with a shadow residual of 0.3052 in the year 2023. This indicates that the USA exhibits a significant level of intermediary importance in the oil-trade network relative to its trade volume, as reflected by its shadow rank of 1.0 and a betweenness score of 0.4859, suggesting a central role in trade relationships [C1].\n\nCitations:\n- [C1] Top shadow hub across all years | Neo4j SHADOW_HUB relationship | all years, rows=1\n- [V1] Vector hit shadow::GBR::2024 | shadow_metric | score=0.621\n- [V2] Vector hit shadow::GBR::2022 | shadow_metric | score=0.621\n- [V3] Vector hit shadow::GBR::2019 | shadow_metric | score=0.621\n- [V4] Vector hit shadow::GBR::2023 | shadow_metric | score=0.620\n- [V5] Vector hit shadow::GBR::2020 | shadow_metric | score=0.618\n- [V6] Vector hit shadow::GBR::2021 | shadow_metric | score=0.615',
 'plan': {'intent': 'top_shadow_hubs_al

In [8]:
ask_graphrag("Which OFAC-exposed countries are also shadow hubs?")

/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QuestionPlan(intent='sanc...s that are shadow hubs'), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: divide by zero encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: overflow encountered in matmul
  sims = (self.matrix @ q_vec) / denom
/Users/nikhil/Downloads/shadow-hubs-graphrag-stiles/GraphRAG/graphrag_helpers/graphrag_langgraph.py:318: RuntimeWarning: invalid value encountered in matmul
  sims = (self.matrix @ q_vec) / denom


Based on the provided evidence, the following OFAC-exposed countries also exhibit characteristics of shadow hubs:

1. **USA**: 
   - **OFAC Entities**: 75
   - **Start Year**: 2019
   - **End Year**: 2024
   - **Latest Shadow Rank**: 1.0
   - **Latest Betweenness**: 0.3495
   - **Shadow Residual**: Increased from 0.1856 in 2019 to 0.2243 in 2024, indicating strong intermediary hub behavior over time [C1].

2. **Italy**: 
   - **OFAC Entities**: 78
   - **Start Year**: 2019
   - **End Year**: 2024
   - **Latest Shadow Rank**: 15.0
   - **Latest Betweenness**: 0.0610
   - **Shadow Residual**: Increased from -0.0341 in 2019 to 0.0123 in 2024, suggesting a trend towards becoming more central in the network [C1].

3. **Belgium**: 
   - **OFAC Entities**: 38
   - **Start Year**: 2019
   - **End Year**: 2024
   - **Latest Shadow Rank**: 34.0
   - **Latest Betweenness**: 0.0386
   - **Shadow Residual**: Increased from -0.0449 in 2019 to -0.0020 in 2024, indicating a trend towards increased str

/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QualityReport(is_sufficie...ne, retrieval_gap=False), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(


{'answer': 'Based on the provided evidence, the following OFAC-exposed countries also exhibit characteristics of shadow hubs:\n\n1. **USA**: \n   - **OFAC Entities**: 75\n   - **Start Year**: 2019\n   - **End Year**: 2024\n   - **Latest Shadow Rank**: 1.0\n   - **Latest Betweenness**: 0.3495\n   - **Shadow Residual**: Increased from 0.1856 in 2019 to 0.2243 in 2024, indicating strong intermediary hub behavior over time [C1].\n\n2. **Italy**: \n   - **OFAC Entities**: 78\n   - **Start Year**: 2019\n   - **End Year**: 2024\n   - **Latest Shadow Rank**: 15.0\n   - **Latest Betweenness**: 0.0610\n   - **Shadow Residual**: Increased from -0.0341 in 2019 to 0.0123 in 2024, suggesting a trend towards becoming more central in the network [C1].\n\n3. **Belgium**: \n   - **OFAC Entities**: 38\n   - **Start Year**: 2019\n   - **End Year**: 2024\n   - **Latest Shadow Rank**: 34.0\n   - **Latest Betweenness**: 0.0386\n   - **Shadow Residual**: Increased from -0.0449 in 2019 to -0.0020 in 2024, indi

In [9]:
ask_graphrag("Give top 5 sanctioned countries that are shadow hubs and have been growing in that activity in the last few years.")

/opt/anaconda3/envs/pythonproject/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=QuestionPlan(intent='sanc...th increasing activity'), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(


Based on the provided evidence, the top five sanctioned countries that exhibit increasing activity as shadow hubs in the oil-trade network from 2019 to 2024 are:

1. **Italy (ITA)**
   - Growth in shadow activity: 0.0464
   - Latest shadow rank: 15.0

2. **Belgium (BEL)**
   - Growth in shadow activity: 0.0429
   - Latest shadow rank: 34.0

3. **USA (USA)**
   - Growth in shadow activity: 0.0388
   - Latest shadow rank: 1.0

4. **Angola (AGO)**
   - Growth in shadow activity: 0.0290
   - Latest shadow rank: 52.0

5. **Colombia (COL)**
   - Growth in shadow activity: 0.0288
   - Latest shadow rank: 42.0

These countries have shown a trend of increasing intermediary importance in the oil-trade network, as indicated by their growth metrics and shadow ranks over the specified period [C1].

Citations:
- [C1] OFAC-exposed shadow hubs with increasing residuals | Neo4j SHADOW_HUB relationship with OFAC overlay | top_n=5
- [V1] Vector hit shadow::HKG::2024 | shadow_metric | score=0.544
- [V2] V

{'answer': 'Based on the provided evidence, the top five sanctioned countries that exhibit increasing activity as shadow hubs in the oil-trade network from 2019 to 2024 are:\n\n1. **Italy (ITA)**\n   - Growth in shadow activity: 0.0464\n   - Latest shadow rank: 15.0\n\n2. **Belgium (BEL)**\n   - Growth in shadow activity: 0.0429\n   - Latest shadow rank: 34.0\n\n3. **USA (USA)**\n   - Growth in shadow activity: 0.0388\n   - Latest shadow rank: 1.0\n\n4. **Angola (AGO)**\n   - Growth in shadow activity: 0.0290\n   - Latest shadow rank: 52.0\n\n5. **Colombia (COL)**\n   - Growth in shadow activity: 0.0288\n   - Latest shadow rank: 42.0\n\nThese countries have shown a trend of increasing intermediary importance in the oil-trade network, as indicated by their growth metrics and shadow ranks over the specified period [C1].\n\nCitations:\n- [C1] OFAC-exposed shadow hubs with increasing residuals | Neo4j SHADOW_HUB relationship with OFAC overlay | top_n=5\n- [V1] Vector hit shadow::HKG::2024 

In [ ]:
# Close when finished:
rag.close()